# Phase 3 — SHACL data-quality validation

SHACL checks that facts are well-formed (the semantic-validity guardrail).
First we validate the good seed data (should PASS), then we inject a broken fact (should FAIL with a clear message).

In [ ]:
from rdflib import Graph
from pyshacl import validate

onto   = Graph().parse('../ontology-ssc/ssc.ttl', format='turtle')
shapes = Graph().parse('../ontology-ssc/shapes/ssc-shapes.ttl', format='turtle')

# 1) Validate the GOOD seed data
good = Graph().parse('../data/seed.ttl', format='turtle')
conforms, _, text = validate(good, shacl_graph=shapes, ont_graph=onto, inference='none')
print('Seed data conforms:', conforms)
print(text)

In [3]:
# 2) Inject a BROKEN fact: a component with no version and no identifier
bad_ttl = '''
@prefix ssc: <http://systemscyber.colostate.edu/ssc#> .
@prefix :    <http://systemscyber.colostate.edu/ssc/data#> .
:broken_component a ssc:SoftwareComponent .
'''
bad = Graph().parse('../data/seed.ttl', format='turtle')
bad.parse(data=bad_ttl, format='turtle')

conforms, _, text = validate(bad, shacl_graph=shapes, ont_graph=onto, inference='none')
print('Broken data conforms:', conforms)
print(text)

Broken data conforms: False
Validation Report
Conforms: False
Results (2):
Constraint Violation in MinCountConstraintComponent (http://www.w3.org/ns/shacl#MinCountConstraintComponent):
	Severity: sh:Violation
	Source Shape: [ sh:datatype xsd:string ; sh:message Literal("Every software component must have a version.") ; sh:minCount Literal("1", datatype=xsd:integer) ; sh:path ssc:hasVersion ]
	Focus Node: :broken_component
	Result Path: ssc:hasVersion
	Message: Every software component must have a version.
Constraint Violation in OrConstraintComponent (http://www.w3.org/ns/shacl#OrConstraintComponent):
	Severity: sh:Violation
	Source Shape: ssc:SoftwareComponentShape
	Focus Node: :broken_component
	Value Node: :broken_component
	Message: Every software component must have a purl or cpe identifier.



## What just happened
- The clean seed **passed** (`conforms: True`).
- The `:broken_component` (no version, no identifier) **failed**, and SHACL told you exactly why.
- In the full pipeline this runs on every incoming fact set (build metadata, intelligence, and any LLM-proposed fact); anything that fails is quarantined rather than trusted — that is the `semantic validity` metric in the evaluation.